In [ ]:
# 1.1 Import required libraries.
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support
from collections import Counter
import time
import warnings
warnings.filterwarnings('ignore')

# Configure fonts for Chinese display text.
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 1.2 Load data.
print("正在加载数据...")
data = pd.read_pickle('pku_labels_train.pkl')

print("数据基本信息:")
print(f"样本数: {len(data)}")
print(f"特征数: {len(data.columns)}")
print("\n列名和数据类型:")
print(data.dtypes)
print(f"\n缺失值统计:\n{data.isnull().sum()}")

In [ ]:
# 2.1 Human-label distribution
print("=== 人类标签分布 ===")
label_counts = data['label_result'].value_counts()
print(label_counts)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Visualize the human-label distribution.
axes[0].bar(label_counts.index, label_counts.values)
axes[0].set_title('人类标注分布')
axes[0].set_xlabel('标签类型')
axes[0].set_ylabel('数量')
for i, v in enumerate(label_counts.values):
    axes[0].text(i, v, str(v), ha='center', va='bottom')

# 2.2 AI-label distributions
ai_labels = ['Citation_Type_GPT', 'Citation_Type_Gemini', 
             'Citation_Type_Claude', 'Citation_Type_DeepSeek']

ai_label_counts = {}
for ai in ai_labels:
    ai_label_counts[ai] = data[ai].value_counts()

# Plot AI-label distributions.
ai_df = pd.DataFrame(ai_label_counts)
ai_df.plot(kind='bar', ax=axes[1])
axes[1].set_title('AI标注分布')
axes[1].set_xlabel('标签类型')
axes[1].set_ylabel('数量')
axes[1].legend(title='AI模型')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# 2.3 Agreement between AI and human labels
print("\n=== AI与人类标注一致性分析 ===")
consistency_scores = {}
for ai in ai_labels:
    agreement = (data[ai] == data['label_result']).mean()
    consistency_scores[ai] = agreement
    print(f"{ai} 与人类标注一致性: {agreement:.2%}")

average_consistency = np.mean(list(consistency_scores.values()))
print(f"\n平均一致性: {average_consistency:.2%}")

# Visualize agreement.
plt.figure(figsize=(8, 5))
ai_names = [ai.replace('Citation_Type_', '') for ai in consistency_scores.keys()]
plt.bar(ai_names, consistency_scores.values())
plt.title('AI模型与人类标注一致性')
plt.xlabel('AI模型')
plt.ylabel('一致性比例')
plt.ylim(0, 1)
for i, (ai, score) in enumerate(consistency_scores.items()):
    plt.text(i, score + 0.01, f"{score:.2%}", ha='center')
plt.show()

In [ ]:
# 3.1 Basic statistics by annotator
print("=== 开始用户可信度分析 ===")

user_stats = data.groupby('user_id').agg({
    'label_result': [
        ('标注数量', 'count'),
        ('标签种类数', lambda x: x.nunique()),
        ('最常用标签', lambda x: x.mode().iloc[0] if not x.mode().empty else None),
        ('最常用标签比例', lambda x: x.value_counts().iloc[0] / len(x) if len(x) > 0 else 0)
    ]
}).reset_index()

user_stats.columns = ['user_id', '标注数量', '标签种类数', '最常用标签', '最常用标签比例']

# 3.2 Agreement between each annotator and AI labels
user_ai_agreement = []
for user_id in data['user_id'].unique():
    user_data = data[data['user_id'] == user_id]
    
    ai_agreements = []
    for ai in ai_labels:
        agreement = (user_data[ai] == user_data['label_result']).mean()
        ai_agreements.append(agreement)
    
    avg_agreement = np.mean(ai_agreements) if ai_agreements else 0
    
    user_ai_agreement.append({
        'user_id': user_id,
        '平均AI一致性': avg_agreement,
        '与GPT一致性': ai_agreements[0] if len(ai_agreements) > 0 else 0,
        '与Gemini一致性': ai_agreements[1] if len(ai_agreements) > 1 else 0,
        '与Claude一致性': ai_agreements[2] if len(ai_agreements) > 2 else 0,
        '与DeepSeek一致性': ai_agreements[3] if len(ai_agreements) > 3 else 0
    })

user_ai_df = pd.DataFrame(user_ai_agreement)
user_stats = pd.merge(user_stats, user_ai_df, on='user_id', how='left')

# 3.3 Calculate annotator credibility scores.
def calculate_credibility_score(row):
    score = 0
    
    # Factor 1: annotation count.
    count_score = min(30, np.log1p(row['标注数量']) * 5)
    score += count_score
    
    # Factor 2: label diversity.
    if row['标签种类数'] == 1:
        diversity_score = 5
    elif row['标签种类数'] == 2:
        diversity_score = 15
    elif row['标签种类数'] == 3:
        diversity_score = 20
    else:  # Four types
        diversity_score = 10
    score += diversity_score
    
    # Factor 3: agreement with AI labels.
    ai_consistency_score = row['平均AI一致性'] * 30
    score += ai_consistency_score
    
    # Factor 4: annotation-pattern consistency.
    if row['最常用标签比例'] < 0.3:
        pattern_score = 5
    elif row['最常用标签比例'] < 0.6:
        pattern_score = 20
    elif row['最常用标签比例'] < 0.8:
        pattern_score = 15
    else:
        pattern_score = 5
    score += pattern_score
    
    return score

user_stats['可信度分数'] = user_stats.apply(calculate_credibility_score, axis=1)
user_stats['可信度等级'] = pd.qcut(user_stats['可信度分数'], q=4, 
                                  labels=['低', '中', '高', '很高'])

print("用户可信度分析结果（前20名）:")
print(user_stats.sort_values('可信度分数', ascending=False).head(20))

In [ ]:
# 4.1 Visualize credibility analysis.
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Panel 1: credibility distribution.
axes[0, 0].hist(user_stats['可信度分数'], bins=30, alpha=0.7, color='skyblue', edgecolor='black')
axes[0, 0].axvline(user_stats['可信度分数'].mean(), color='red', linestyle='--', label=f'平均: {user_stats["可信度分数"].mean():.1f}')
axes[0, 0].axvline(user_stats['可信度分数'].median(), color='green', linestyle='--', label=f'中位数: {user_stats["可信度分数"].median():.1f}')
axes[0, 0].set_title('用户可信度分数分布')
axes[0, 0].set_xlabel('可信度分数')
axes[0, 0].set_ylabel('用户数量')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Panel 2: credibility versus annotation count.
axes[0, 1].scatter(user_stats['标注数量'], user_stats['可信度分数'], 
                   alpha=0.6, c=user_stats['平均AI一致性'], cmap='viridis')
axes[0, 1].set_xscale('log')
axes[0, 1].set_title('标注数量 vs 可信度分数')
axes[0, 1].set_xlabel('标注数量（对数尺度）')
axes[0, 1].set_ylabel('可信度分数')
axes[0, 1].grid(True, alpha=0.3)

# Panel 3: credibility versus agreement with AI labels.
scatter = axes[1, 0].scatter(user_stats['平均AI一致性'], user_stats['可信度分数'],
                            c=user_stats['标注数量'], cmap='plasma', alpha=0.7, s=50)
axes[1, 0].set_title('AI一致性 vs 可信度分数')
axes[1, 0].set_xlabel('平均AI一致性')
axes[1, 0].set_ylabel('可信度分数')
axes[1, 0].grid(True, alpha=0.3)
plt.colorbar(scatter, ax=axes[1, 0], label='标注数量')

# Panel 4: credibility-level distribution.
credibility_counts = user_stats['可信度等级'].value_counts().sort_index()
colors = ['lightcoral', 'gold', 'lightgreen', 'skyblue']
axes[1, 1].bar(credibility_counts.index, credibility_counts.values, color=colors)
axes[1, 1].set_title('可信度等级分布')
axes[1, 1].set_xlabel('可信度等级')
axes[1, 1].set_ylabel('用户数量')
for i, (level, count) in enumerate(credibility_counts.items()):
    axes[1, 1].text(i, count + 0.5, str(count), ha='center')

plt.tight_layout()
plt.show()

# 4.2 Identify potentially unreliable annotators.
unreliable_criteria = (
    (user_stats['平均AI一致性'] < 0.3) |  # Low agreement with AI labels
    (user_stats['标签种类数'] == 1) |      # Only one label type
    (user_stats['最常用标签比例'] > 0.7) |  # Excessive label concentration
    (user_stats['标注数量'] < 10)          # Too few annotations
)

unreliable_users = user_stats[unreliable_criteria]
print(f"\n识别到 {len(unreliable_users)} 个潜在不可信用户")
print("潜在不可信用户示例:")
print(unreliable_users[['user_id', '标注数量', '标签种类数', '最常用标签比例', '平均AI一致性']].head(20))

In [ ]:
# 5.1 Select high-credibility annotators.
print("\n=== 根据可信度过滤数据 ===")
threshold_score = user_stats['可信度分数'].quantile(0.5)
high_cred_users = user_stats[user_stats['可信度分数'] >= threshold_score]

print(f"高可信度用户阈值分数: {threshold_score:.1f}")
print(f"高可信度用户数量: {len(high_cred_users)} (占总用户 {len(high_cred_users)/len(user_stats)*100:.1f}%)")

# 5.2 Filter the data.
filtered_data = data[data['user_id'].isin(high_cred_users['user_id'])]
print(f"过滤后数据量: {len(filtered_data)} (占总数据 {len(filtered_data)/len(data)*100:.1f}%)")

# 5.3 Compare label distributions before and after filtering.
print("\n=== 过滤前后标签分布比较 ===")
original_dist = data['label_result'].value_counts(normalize=True)
filtered_dist = filtered_data['label_result'].value_counts(normalize=True)

dist_comparison = pd.DataFrame({
    '原始分布': original_dist,
    '过滤后分布': filtered_dist,
    '变化': filtered_dist - original_dist
})
print(dist_comparison)

# Visualize distribution changes.
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(original_dist))
width = 0.35

ax.bar(x - width/2, original_dist.values, width, label='原始分布', color='lightblue')
ax.bar(x + width/2, filtered_dist.values, width, label='过滤后分布', color='lightgreen')
ax.set_xlabel('标签类型')
ax.set_ylabel('比例')
ax.set_title('过滤前后标签分布比较')
ax.set_xticks(x)
ax.set_xticklabels(original_dist.index)
ax.legend()

# Add value labels.
for i, v in enumerate(original_dist.values):
    ax.text(i - width/2, v + 0.01, f'{v:.3f}', ha='center')
for i, v in enumerate(filtered_dist.values):
    ax.text(i + width/2, v + 0.01, f'{v:.3f}', ha='center')

plt.tight_layout()
plt.show()

In [ ]:
# 6.1 Use 1024-dimensional vectors.
print("\n=== 改进的特征工程 ===")
X = np.array(filtered_data['embedding_vector'].tolist())
y = filtered_data['label_result'].values

print(f"基础特征维度: {X.shape}")

# 6.2 Add further features.
# 6.2.1 Paragraph-length features
filtered_data['para_length'] = filtered_data['Para_text'].apply(len)
# 6.2.2 Citation-format features
filtered_data['has_brackets'] = filtered_data['Ref_Clean_citation'].str.contains(r'[\[\]()]', na=False)
filtered_data['has_digits'] = filtered_data['Ref_Clean_citation'].str.contains(r'\d', na=False)
# 6.2.3 One-hot encoding of AI labels
ai_features = pd.get_dummies(filtered_data[ai_labels].applymap(lambda x: str(x)), 
                             prefix_sep='_', dummy_na=False)

# 6.3 Combine all features.
# Numeric features
numeric_features = filtered_data[['para_length']].copy()
numeric_features['has_brackets'] = filtered_data['has_brackets'].astype(int)
numeric_features['has_digits'] = filtered_data['has_digits'].astype(int)

# Normalize numeric features.
scaler = StandardScaler()
numeric_features_scaled = scaler.fit_transform(numeric_features)

# Combine all features.
X_enhanced = np.hstack([X, numeric_features_scaled, ai_features.values])

print(f"原始特征维度: {X.shape}")
print(f"增强后特征维度: {X_enhanced.shape}")
print(f"特征构成:")
print(f"  - 1024维文本向量")
print(f"  - {numeric_features_scaled.shape[1]}个数值特征")
print(f"  - {ai_features.shape[1]}个AI标签特征")

# 6.4 Encode labels.
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
print(f"\n标签编码映射: {dict(enumerate(label_encoder.classes_))}")

In [ ]:
# 7.1 Split by annotator so train and test users do not overlap.
print("\n=== 改进的数据划分（按用户划分） ===")

unique_users = filtered_data['user_id'].unique()
n_users = len(unique_users)
n_train_users = int(n_users * 0.7)

# Randomly select training annotators.
np.random.seed(42)
train_users = np.random.choice(unique_users, n_train_users, replace=False)
test_users = np.array([u for u in unique_users if u not in train_users])

# Split observations by annotator.
train_mask = filtered_data['user_id'].isin(train_users)
test_mask = filtered_data['user_id'].isin(test_users)

X_train = X_enhanced[train_mask]
X_test = X_enhanced[test_mask]
y_train = y_encoded[train_mask]
y_test = y_encoded[test_mask]

# Save the split data for later analysis.
train_data = filtered_data[train_mask].copy()
test_data = filtered_data[test_mask].copy()

print(f"训练用户数: {len(train_users)}")
print(f"测试用户数: {len(test_users)}")
print(f"训练集大小: {len(X_train)} (占过滤后数据 {len(X_train)/len(filtered_data)*100:.1f}%)")
print(f"测试集大小: {len(X_test)} (占过滤后数据 {len(X_test)/len(filtered_data)*100:.1f}%)")

# 7.2 Check label distributions.
train_label_dist = pd.Series(y_train).value_counts().sort_index()
test_label_dist = pd.Series(y_test).value_counts().sort_index()

print("\n训练集标签分布:")
for i, (label, count) in enumerate(zip(label_encoder.classes_, train_label_dist)):
    print(f"  {label}: {count} ({count/len(y_train)*100:.1f}%)")

print("\n测试集标签分布:")
for i, (label, count) in enumerate(zip(label_encoder.classes_, test_label_dist)):
    print(f"  {label}: {count} ({count/len(y_test)*100:.1f}%)")

# 7.3 Visualize dataset distributions.
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Training-set distribution
axes[0].bar(label_encoder.classes_, train_label_dist.values, color='skyblue')
axes[0].set_title('训练集标签分布')
axes[0].set_xlabel('标签类型')
axes[0].set_ylabel('数量')
axes[0].tick_params(axis='x', rotation=45)
for i, v in enumerate(train_label_dist.values):
    axes[0].text(i, v, str(v), ha='center', va='bottom')

# Test-set distribution
axes[1].bar(label_encoder.classes_, test_label_dist.values, color='lightgreen')
axes[1].set_title('测试集标签分布')
axes[1].set_xlabel('标签类型')
axes[1].set_ylabel('数量')
axes[1].tick_params(axis='x', rotation=45)
for i, v in enumerate(test_label_dist.values):
    axes[1].text(i, v, str(v), ha='center', va='bottom')

plt.tight_layout()
plt.show()

In [ ]:
# 8.1 Import model libraries.
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

# 8.2 Define models.
models = {
    '逻辑回归': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    '随机森林': RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced', n_jobs=-1),
    '额外树': ExtraTreesClassifier(n_estimators=200, random_state=42, class_weight='balanced', n_jobs=-1),
    '支持向量机': SVC(random_state=42, class_weight='balanced', probability=True),
    '神经网络': MLPClassifier(hidden_layer_sizes=(512, 256, 128), max_iter=1000, random_state=42, 
                            early_stopping=True, validation_fraction=0.1),
    'XGBoost': XGBClassifier(n_estimators=200, random_state=42, 
                           use_label_encoder=False, eval_metric='mlogloss', n_jobs=-1)
}

# 8.3 Train and evaluate models.
print("\n=== 开始模型训练 ===")
results = {}

for name, model in models.items():
    print(f"\n=== 训练 {name} ===")
    start_time = time.time()
    
    try:
        # Train the model.
        model.fit(X_train, y_train)
        
        # Predict.
        y_pred = model.predict(X_test)
        y_pred_proba = model.predict_proba(X_test) if hasattr(model, "predict_proba") else None
        
        # Calculate metrics.
        accuracy = accuracy_score(y_test, y_pred)
        train_accuracy = accuracy_score(y_train, model.predict(X_train))
        precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted')
        
        # Save results.
        results[name] = {
            'model': model,
            'y_pred': y_pred,
            'y_pred_proba': y_pred_proba,
            'test_accuracy': accuracy,
            'train_accuracy': train_accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'training_time': time.time() - start_time
        }
        
        print(f"训练准确率: {train_accuracy:.4f}")
        print(f"测试准确率: {accuracy:.4f}")
        print(f"F1分数: {f1:.4f}")
        print(f"训练时间: {results[name]['training_time']:.2f}秒")
        
        # Detailed classification report
        print("\n分类报告:")
        print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))
        
    except Exception as e:
        print(f"训练模型 {name} 时出错: {str(e)}")

In [ ]:
# 9.1 Compare models.
if results:
    print("\n=== 模型性能比较 ===")
    comparison_df = pd.DataFrame({
        'Model': list(results.keys()),
        '训练准确率': [results[m]['train_accuracy'] for m in results],
        '测试准确率': [results[m]['test_accuracy'] for m in results],
        'F1分数': [results[m]['f1'] for m in results],
        '训练时间(秒)': [results[m]['training_time'] for m in results]
    })

    print(comparison_df.sort_values('测试准确率', ascending=False))

    # 9.2 Visualize model comparisons.
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Accuracy comparison
    comparison_df.set_index('Model')[['训练准确率', '测试准确率']].plot(kind='bar', ax=axes[0, 0])
    axes[0, 0].set_title('模型准确率比较')
    axes[0, 0].set_ylabel('准确率')
    axes[0, 0].set_ylim(0, 1)
    axes[0, 0].legend(loc='lower right')
    axes[0, 0].tick_params(axis='x', rotation=45)
    
    # F1 comparison
    axes[0, 1].bar(comparison_df['Model'], comparison_df['F1分数'])
    axes[0, 1].set_title('模型F1分数比较')
    axes[0, 1].set_ylabel('F1分数')
    axes[0, 1].set_ylim(0, 1)
    axes[0, 1].tick_params(axis='x', rotation=45)
    for i, (model, f1) in enumerate(zip(comparison_df['Model'], comparison_df['F1分数'])):
        axes[0, 1].text(i, f1 + 0.01, f'{f1:.3f}', ha='center', fontsize=9)
    
    # Training-time comparison
    axes[1, 0].bar(comparison_df['Model'], comparison_df['训练时间(秒)'])
    axes[1, 0].set_title('模型训练时间比较')
    axes[1, 0].set_ylabel('时间(秒)')
    axes[1, 0].tick_params(axis='x', rotation=45)
    
    # Degree of overfitting
    comparison_df['过拟合程度'] = comparison_df['训练准确率'] - comparison_df['测试准确率']
    axes[1, 1].bar(comparison_df['Model'], comparison_df['过拟合程度'])
    axes[1, 1].set_title('模型过拟合程度（训练-测试准确率差）')
    axes[1, 1].set_ylabel('过拟合程度')
    axes[1, 1].tick_params(axis='x', rotation=45)
    axes[1, 1].axhline(y=0, color='r', linestyle='-', alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# 9.3 Analyze the best model.
if results:
    best_model_name = comparison_df.loc[comparison_df['测试准确率'].idxmax(), 'Model']
    best_model = results[best_model_name]
    print(f"\n=== 最佳模型: {best_model_name} ===")
    print(f"测试准确率: {best_model['test_accuracy']:.4f}")
    print(f"F1分数: {best_model['f1']:.4f}")
    
    # Confusion matrix
    cm = confusion_matrix(y_test, best_model['y_pred'])
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=label_encoder.classes_, 
                yticklabels=label_encoder.classes_)
    plt.title(f'{best_model_name} 混淆矩阵')
    plt.xlabel('预测标签')
    plt.ylabel('真实标签')
    plt.show()

In [ ]:
# Section 9.5: save the model and create prediction functions.
print("\n=== 保存模型和创建预测功能 ===")

import joblib
import json
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# 9.5.1 Identify the best model.
print("1. 确定最佳模型...")
if 'results' in locals() and len(results) > 0:
    # Select the best model from earlier results.
    best_model_name = "神经网络"  # This model was selected from the recorded output.
    if best_model_name in results:
        best_model = results[best_model_name]
        print(f"已选择最佳模型: {best_model_name}")
        print(f"测试准确率: {best_model['test_accuracy']:.4f}")
        print(f"F1分数: {best_model['f1']:.4f}")
    else:
        print(f"警告: 未找到模型 '{best_model_name}'，将使用第一个可用模型")
        best_model_name = list(results.keys())[0]
        best_model = results[best_model_name]
        print(f"使用模型: {best_model_name}")
else:
    print("错误: 未找到训练结果，请先运行模型训练")
    raise ValueError("未找到训练结果")

# 9.5.2 Create the output directory.
save_dir = "saved_models"
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
    print(f"创建保存目录: {save_dir}")

# 9.5.3 Save the best model.
print(f"\n2. 保存最佳模型 '{best_model_name}'...")
model_filename = os.path.join(save_dir, f"best_model_{datetime.now().strftime('%Y%m%d_%H%M%S')}.pkl")
joblib.dump(best_model['model'], model_filename)
print(f"最佳模型已保存到: {model_filename}")

# 9.5.4 Save the label encoder.
label_encoder_filename = os.path.join(save_dir, "label_encoder.pkl")
joblib.dump(label_encoder, label_encoder_filename)
print(f"标签编码器已保存到: {label_encoder_filename}")

# 9.5.5 Save the feature scaler.
scaler_filename = os.path.join(save_dir, "feature_scaler.pkl")
joblib.dump(scaler, scaler_filename)
print(f"特征缩放器已保存到: {scaler_filename}")

# 9.5.6 Save AI-label encoding metadata.
ai_features_info = {
    'ai_labels': ai_labels,
    'ai_features_columns': ai_features.columns.tolist(),
    'ai_features_shape': ai_features.shape
}
ai_features_filename = os.path.join(save_dir, "ai_features_info.json")
with open(ai_features_filename, 'w', encoding='utf-8') as f:
    json.dump(ai_features_info, f, ensure_ascii=False, indent=2)
print(f"AI特征信息已保存到: {ai_features_filename}")

# 9.5.7 Save predictions.
print("\n3. 保存预测结果...")
test_data_with_pred = test_data.copy()
test_data_with_pred['预测标签'] = label_encoder.inverse_transform(best_model['y_pred'])
test_data_with_pred['真实标签'] = label_encoder.inverse_transform(y_test)

# Add prediction probabilities.
if best_model['y_pred_proba'] is not None:
    test_data_with_pred['预测概率'] = np.max(best_model['y_pred_proba'], axis=1)
    
    # Add a probability for each class.
    for i, label in enumerate(label_encoder.classes_):
        test_data_with_pred[f'预测概率_{label}'] = best_model['y_pred_proba'][:, i]

# Record whether each prediction is correct.
test_data_with_pred['预测正确'] = test_data_with_pred['预测标签'] == test_data_with_pred['真实标签']

# Save to CSV.
predictions_filename = os.path.join(save_dir, "predictions.csv")
test_data_with_pred[['ID', 'Para_text', 'Ref_Clean_citation', 'user_id', 
                     '真实标签', '预测标签', '预测正确', '预测概率'] + 
                    [f'预测概率_{label}' for label in label_encoder.classes_]].to_csv(
    predictions_filename, index=False, encoding='utf-8-sig')
print(f"预测结果已保存到: {predictions_filename}")

# 9.5.8 Save the model-performance report.
print("\n4. 保存模型性能报告...")
performance_report = {
    '模型信息': {
        '模型名称': best_model_name,
        '保存时间': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        '模型文件': model_filename
    },
    '数据信息': {
        '训练集大小': len(X_train),
        '测试集大小': len(X_test),
        '特征维度': X_train.shape[1],
        '标签类别': label_encoder.classes_.tolist()
    },
    '性能指标': {
        '训练准确率': float(best_model['train_accuracy']),
        '测试准确率': float(best_model['test_accuracy']),
        'F1分数': float(best_model['f1']),
        '训练时间(秒)': float(best_model['training_time'])
    },
    '数据预处理信息': {
        '原始数据量': len(data),
        '过滤后数据量': len(filtered_data),
        '高可信度用户数': len(high_cred_users),
        '特征构成': {
            '文本向量维度': 1024,
            '数值特征维度': numeric_features_scaled.shape[1],
            'AI标签特征维度': ai_features.shape[1],
            '总特征维度': X_enhanced.shape[1]
        }
    }
}

# Add the confusion matrix.
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, best_model['y_pred'])
performance_report['混淆矩阵'] = {
    'labels': label_encoder.classes_.tolist(),
    'matrix': cm.tolist()
}

# Add per-class F1 scores.
from sklearn.metrics import f1_score
f1_scores = f1_score(y_test, best_model['y_pred'], average=None, zero_division=0)
performance_report['各类别F1分数'] = {label: float(score) for label, score in zip(label_encoder.classes_, f1_scores)}

# Save the report.
report_filename = os.path.join(save_dir, "model_performance_report.json")
with open(report_filename, 'w', encoding='utf-8') as f:
    json.dump(performance_report, f, ensure_ascii=False, indent=2)
print(f"模型性能报告已保存到: {report_filename}")

# 9.5.9 Create a function to predict new observations.
print("\n5. 创建预测新数据的函数...")

def predict_new_data(new_data, model_path=model_filename, 
                    label_encoder_path=label_encoder_filename,
                    scaler_path=scaler_filename,
                    ai_features_info_path=ai_features_filename,
                    return_proba=True):
    """
    Predict citation types for new observations.

    Parameters
    ----------
    new_data : pandas.DataFrame
        Required fields: Para_text (paragraph), Ref_Clean_citation (cleaned citation),
        and embedding_vector (required 1024-dimensional vector).
        Optional AI-label columns: Citation_Type_GPT, Citation_Type_Gemini,
        Citation_Type_Claude, and Citation_Type_DeepSeek.
    model_path : str
        Model file path.
    label_encoder_path : str
        Label-encoder path.
    scaler_path : str
        Feature-scaler path.
    ai_features_info_path : str
        AI-feature metadata path.
    return_proba : bool
        Whether to return a probability for each class.

    Returns
    -------
    pandas.DataFrame
        Prediction results.
    """
    
    print(f"开始预测 {len(new_data)} 条新数据...")
    
    # 1. Load the model and supporting components.
    model = joblib.load(model_path)
    label_encoder = joblib.load(label_encoder_path)
    scaler = joblib.load(scaler_path)
    
    with open(ai_features_info_path, 'r', encoding='utf-8') as f:
        ai_info = json.load(f)
    
    # 2. Check required columns.
    required_cols = ['Para_text', 'Ref_Clean_citation', 'embedding_vector']
    for col in required_cols:
        if col not in new_data.columns:
            raise ValueError(f"新数据缺少必要的列: '{col}'")
    
    # 3. Extract embedding_vector features.
    X_new = np.array(new_data['embedding_vector'].tolist())
    
    # 4. Extract additional numeric features.
    # Paragraph length
    para_length = new_data['Para_text'].apply(len).values.reshape(-1, 1)
    
    # Citation-format features
    has_brackets = new_data['Ref_Clean_citation'].str.contains(r'[\[\]()]', na=False).fillna(False).astype(int).values.reshape(-1, 1)
    has_digits = new_data['Ref_Clean_citation'].str.contains(r'\d', na=False).fillna(False).astype(int).values.reshape(-1, 1)
    
    # Combine numeric features.
    numeric_features = np.hstack([para_length, has_brackets, has_digits])
    
    # Normalize numeric features.
    numeric_features_scaled = scaler.transform(numeric_features)
    
    # 5. Extract AI-label features.
    # Encode all AI labels once, then align exactly to the saved training schema.
    available = [label for label in ai_info['ai_labels'] if label in new_data.columns]
    if available:
        ai_frame = pd.get_dummies(new_data[available].fillna('Unknown').astype(str),
                                  prefix_sep='_')
    else:
        ai_frame = pd.DataFrame(index=new_data.index)
    ai_features_new = ai_frame.reindex(
        columns=ai_info['ai_features_columns'], fill_value=0
    ).to_numpy(dtype=float)
    
    # 6. Combine all features.
    X_new_enhanced = np.hstack([X_new, numeric_features_scaled, ai_features_new])
    
    # 7. Predict.
    if return_proba and hasattr(model, "predict_proba"):
        y_pred_proba = model.predict_proba(X_new_enhanced)
        y_pred = model.predict(X_new_enhanced)
    else:
        y_pred = model.predict(X_new_enhanced)
        y_pred_proba = None
    
    # 8. Prepare results.
    results = new_data.copy()
    results['预测标签'] = label_encoder.inverse_transform(y_pred)
    
    if y_pred_proba is not None:
        results['预测概率'] = np.max(y_pred_proba, axis=1)
        
        # Add probabilities for each class.
        for i, label in enumerate(label_encoder.classes_):
            results[f'预测概率_{label}'] = y_pred_proba[:, i]
        
        # Add prediction confidence.
        results['预测置信度'] = pd.cut(results['预测概率'], 
                                    bins=[0, 0.5, 0.7, 0.85, 1.0],
                                    labels=['低', '中', '高', '很高'])
    
    print(f"预测完成！")
    print(f"预测标签分布: {results['预测标签'].value_counts().to_dict()}")
    
    return results

# 9.5.10 Create a batch-prediction function.
def batch_predict(input_file, output_file=None, model_dir=save_dir):
    """
    Predict new observations from a file in batches.

    Parameters
    ----------
    input_file : str
        CSV or pickle input file path.
    output_file : str, optional
        Output path; by default append a prediction suffix to the input filename.
    model_dir : str
        Directory containing model components.

    Returns
    -------
    pandas.DataFrame
        Prediction results.
    """
    
    # Find model files.
    model_files = [f for f in os.listdir(model_dir) if f.startswith('best_model_') and f.endswith('.pkl')]
    if not model_files:
        raise FileNotFoundError(f"在目录 '{model_dir}' 中未找到模型文件")
    
    # Use the latest model.
    latest_model = sorted(model_files)[-1]
    model_path = os.path.join(model_dir, latest_model)
    
    # Load input data.
    if input_file.endswith('.csv'):
        new_data = pd.read_csv(input_file, encoding='utf-8')
    elif input_file.endswith('.pkl'):
        new_data = pd.read_pickle(input_file)
    else:
        raise ValueError("只支持.csv和.pkl格式的输入文件")
    
    print(f"加载数据: {input_file} ({len(new_data)} 行)")
    
    # Check whether embedding_vector is required.
    if 'embedding_vector' not in new_data.columns:
        print("警告: 输入数据缺少'embedding_vector'列")
        print("请确保使用BGE-M3模型提取1024维文本向量")
        print("列名应为'embedding_vector'，值为1024维列表")
    
    # Predict.
    predictions = predict_new_data(new_data, model_path=model_path)
    
    # Save results.
    if output_file is None:
        base_name = os.path.splitext(input_file)[0]
        output_file = f"{base_name}_预测结果.csv"
    
    predictions.to_csv(output_file, index=False, encoding='utf-8-sig')
    print(f"预测结果已保存到: {output_file}")
    
    return predictions

# 9.5.11 Create usage examples.
print("\n6. 创建使用示例和说明...")
example_usage = f"""
# ============================================================
# Citation-label prediction model: usage
# Saved at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
# Selected model: {best_model_name}
# Test accuracy: {best_model['test_accuracy']:.4f}
# ============================================================

# Method 1: use the prediction function
# ------------------------------
# Assume new_data is a DataFrame containing these columns:
# - 'Para_text': paragraph text
# - 'Ref_Clean_citation': cleaned citation
# - 'embedding_vector': 1024-dimensional text vector (required)
# - optional AI-label columns

# Load the saved model and functions
import pandas as pd
import numpy as np
import joblib
import json
import os
from sklearn.preprocessing import StandardScaler

# Define the prediction function (copy predict_new_data above)
# ... or import it from saved_models ...

# Predict new observations
predictions = predict_new_data(new_data)
print(predictions[['Para_text', '预测标签', '预测概率', '预测置信度']].head(10))

# Method 2: predict a file in batches
# ------------------------------
# Create example observations
test_example = pd.DataFrame({{
    'Para_text': ['This paper builds on previous work by Smith et al. to develop a new method.'],
    'Ref_Clean_citation': ['Smith et al. (2020)'],
    'embedding_vector': [np.random.randn(1024).tolist()],  # 替换为真实的BGE-M3向量
    'Citation_Type_GPT': ['Background'],
    'Citation_Type_Gemini': ['Technical basis']
}})

# Save the example data
test_example.to_csv('test_data_example.csv', index=False, encoding='utf-8-sig')

# Run batch prediction
predictions = batch_predict('test_data_example.csv', 'test_predictions.csv')

# Method 3: command-line usage
# ------------------------------
# Run from the command line:
# python -c "from saved_models import batch_predict; batch_predict('new_data.csv')"
"""

# Save usage instructions.
usage_filename = os.path.join(save_dir, "使用说明.txt")
with open(usage_filename, 'w', encoding='utf-8') as f:
    f.write(example_usage)
print(f"使用说明已保存到: {usage_filename}")

# 9.5.12 Create a simple deployment script.
deployment_script = f'''#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
引用标签预测模型 - 简易部署脚本
用于预测新数据的引用类型标签
"""

import pandas as pd
import numpy as np
import joblib
import json
import os
import sys
import warnings
warnings.filterwarnings('ignore')

def load_model_components(model_dir='saved_models'):
    """加载模型和所有组件"""
    components = {{}}
    
    # Find the latest model file
    model_files = [f for f in os.listdir(model_dir) if f.startswith('best_model_') and f.endswith('.pkl')]
    if not model_files:
        raise FileNotFoundError(f"在目录 '{{model_dir}}' 中未找到模型文件")
    
    latest_model = sorted(model_files)[-1]
    components['model'] = joblib.load(os.path.join(model_dir, latest_model))
    components['label_encoder'] = joblib.load(os.path.join(model_dir, 'label_encoder.pkl'))
    components['scaler'] = joblib.load(os.path.join(model_dir, 'feature_scaler.pkl'))
    
    with open(os.path.join(model_dir, 'ai_features_info.json'), 'r', encoding='utf-8') as f:
        components['ai_info'] = json.load(f)
    
    return components

def predict_citation_single(text, citation, embedding_vector, ai_labels=None, model_dir='saved_models'):
    """
    预测单条数据的引用类型
    
    参数:
    ----------
    text : str
        段落文本
    citation : str
        清洗后的引用
    embedding_vector : list
        1024维文本向量
    ai_labels : dict, optional
        AI标签，例如 {{'GPT': 'Background', 'Gemini': 'Technical basis'}}
    model_dir : str
        模型目录
    
    返回:
    -------
    dict : 预测结果
    """
    # Create one observation
    data = {{
        'Para_text': [text],
        'Ref_Clean_citation': [citation],
        'embedding_vector': [embedding_vector]
    }}
    
    # Add AI labels
    if ai_labels:
        for ai_name, ai_label in ai_labels.items():
            data[f'Citation_Type_{{ai_name}}'] = [ai_label]
    
    df = pd.DataFrame(data)
    
    # Load components
    components = load_model_components(model_dir)
    
    # Extract features and predict
    result = predict_new_data(df, components)
    
    return result.iloc[0].to_dict()

def predict_new_data(new_data, components=None, model_dir='saved_models'):
    """预测新数据"""
    if components is None:
        components = load_model_components(model_dir)
    
    model = components['model']
    label_encoder = components['label_encoder']
    scaler = components['scaler']
    ai_info = components['ai_info']
    
    # Feature extraction
    X_new = np.array(new_data['embedding_vector'].tolist())
    
    # Numeric features
    para_length = new_data['Para_text'].apply(len).values.reshape(-1, 1)
    has_brackets = new_data['Ref_Clean_citation'].str.contains(r'[\[\]()]', na=False).fillna(False).astype(int).values.reshape(-1, 1)
    has_digits = new_data['Ref_Clean_citation'].str.contains(r'\d', na=False).fillna(False).astype(int).values.reshape(-1, 1)
    
    numeric_features = np.hstack([para_length, has_brackets, has_digits])
    numeric_features_scaled = scaler.transform(numeric_features)
    
    # AI features
    # Encode all AI labels once, then align exactly to the saved training schema.
    available = [label for label in ai_info['ai_labels'] if label in new_data.columns]
    if available:
        ai_frame = pd.get_dummies(new_data[available].fillna('Unknown').astype(str),
                                  prefix_sep='_')
    else:
        ai_frame = pd.DataFrame(index=new_data.index)
    ai_features_new = ai_frame.reindex(
        columns=ai_info['ai_features_columns'], fill_value=0
    ).to_numpy(dtype=float)
    
    # Combine features
    X_new_enhanced = np.hstack([X_new, numeric_features_scaled, ai_features_new])
    
    # Predict
    y_pred_proba = model.predict_proba(X_new_enhanced)
    y_pred = model.predict(X_new_enhanced)
    
    # Prepare results
    results = new_data.copy()
    results['预测标签'] = label_encoder.inverse_transform(y_pred)
    results['预测概率'] = np.max(y_pred_proba, axis=1)
    
    for i, label in enumerate(label_encoder.classes_):
        results[f'预测概率_{{label}}'] = y_pred_proba[:, i]
    
    results['预测置信度'] = pd.cut(results['预测概率'], 
                                bins=[0, 0.5, 0.7, 0.85, 1.0],
                                labels=['低', '中', '高', '很高'])
    
    return results

if __name__ == "__main__":
    if len(sys.argv) < 2:
        print("用法: python citation_predictor.py <输入文件> [输出文件]")
        print("示例: python citation_predictor.py new_data.csv predictions.csv")
        sys.exit(1)
    
    input_file = sys.argv[1]
    output_file = sys.argv[2] if len(sys.argv) > 2 else None
    
    # Load data
    if input_file.endswith('.csv'):
        new_data = pd.read_csv(input_file, encoding='utf-8')
    elif input_file.endswith('.pkl'):
        new_data = pd.read_pickle(input_file)
    else:
        print("错误: 只支持.csv和.pkl格式")
        sys.exit(1)
    
    # Predict
    try:
        predictions = predict_new_data(new_data)
        
        if output_file is None:
            base_name = os.path.splitext(input_file)[0]
            output_file = f"{{base_name}}_预测结果.csv"
        
        predictions.to_csv(output_file, index=False, encoding='utf-8-sig')
        print(f"预测完成! 结果保存到: {{output_file}}")
        
    except Exception as e:
        print(f"预测失败: {{str(e)}}")
        sys.exit(1)
'''

# Save the deployment script.
script_filename = os.path.join(save_dir, "citation_predictor.py")
with open(script_filename, 'w', encoding='utf-8') as f:
    f.write(deployment_script)
print(f"部署脚本已保存到: {script_filename}")

# 9.5.13 Final summary
print("\n" + "="*60)
print("模型保存和部署完成！")
print("="*60)
print(f"保存目录: {save_dir}")
print(f"最佳模型: {best_model_name}")
print(f"测试准确率: {best_model['test_accuracy']:.4f}")
print(f"F1分数: {best_model['f1']:.4f}")

# Print per-class F1 scores.
if 'f1_scores' in performance_report:
    print("\n各类别F1分数:")
    for label, score in performance_report['各类别F1分数'].items():
        star = " *" if label in ['Comparison', 'Fundamental idea'] else ""
        print(f"  {label}{star}: {score:.4f}")

print(f"\n已保存文件:")
print(f"1. 模型文件: {model_filename}")
print(f"2. 标签编码器: {label_encoder_filename}")
print(f"3. 特征缩放器: {scaler_filename}")
print(f"4. AI特征信息: {ai_features_filename}")
print(f"5. 预测结果: {predictions_filename}")
print(f"6. 性能报告: {report_filename}")
print(f"7. 使用说明: {usage_filename}")
print(f"8. 部署脚本: {script_filename}")

print(f"\n使用说明:")
print(f"1. 预测新数据: 使用 predict_new_data() 函数")
print(f"2. 批量预测: python {script_filename} 新数据.csv")
print(f"3. 查看预测结果: 打开 {predictions_filename}")
print(f"4. 查看模型报告: 打开 {report_filename}")
print("="*60)